In [ ]:
# Ciaran Whelan (23370211), Emma O'Reilly (23304472), Hersh Singh (23371137)
# Runs to the end without an error (checked 3 Nov 2026 using the saved runs, about 3 minutes on an RTX 3080)
# YOLOv8 neck, Detect head, loss and NMS reused from Ultralytics: https://github.com/ultralytics/ultralytics

# Fire and Smoke Detection with a ResNet-50 and YOLOv8 Hybrid

CS4287 Assignment 2. The sections follow the PDF report, and each one says who wrote its code.

1. The data set
2. The network structure
3. Loss function, optimiser and cross-fold validation
4. Results and evaluation
5. Independent experiments: hyperparameter tuning

**How it runs.** Every step first looks for its saved output in `results/` (folds, training histories, model weights) and uses it. Anything missing is trained only when `ALLOW_TRAINING = True` in the settings cell, and only the missing parts are trained; otherwise the notebook says what is missing. With everything saved it runs to the end in about 3 minutes; training everything takes several hours on a GPU.

**Setup.** Clone the repo with its `data` and `results` submodules (see `guide.md`), install `requirements.txt`, and run this notebook from `notebooks/final/`.

## Setup

*Code: Hersh Singh*

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torchmetrics.detection import MeanAveragePrecision
from ultralytics.utils.nms import non_max_suppression

import json
import warnings
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import stats

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

# same folders as src/config/paths.py, built from ROOT as a notebook has no __file__
DATASET_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
CSV_DIR = RESULTS_DIR / "csvs"
IMAGE_DIR = RESULTS_DIR / "images"
PLOT_DIR = RESULTS_DIR / "plots"
MODEL_DIR = RESULTS_DIR / "models"
WEIGHTS_DIR = MODEL_DIR / "weights"
SAVED_MODELS = RESULTS_DIR / "models" # MODEL_DIR is moved while saving runs, this stays put

# the team's code from src/, replaced by the definitions pasted into sections 1 to 3
from src.fine_tune import Trainer, HybridModel
from src.fine_tune.losses import YOLOLoss
from src.data.preprocessing import ModelData, train_transform, eval_transform

from src.utils import parse_boxes, collate_fn, print_evaluation
from src.config.types import TrainingObjects

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# train_model logs CUDA memory but Mac doesnt support; ignore errors
if device.type != "cuda":
    torch.cuda.reset_peak_memory_stats = lambda: None
    torch.cuda.max_memory_allocated = lambda: 0

warnings.filterwarnings("ignore", message="CUDA is not available")
warnings.filterwarnings("ignore", message="Encountered more than 100 detections")

git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()


def save_runs_to(folder):
    # Trainer.save_results writes to MODEL_DIR of the module Trainer is defined in (src/ now, this notebook once pasted)
    sys.modules[Trainer.__module__].MODEL_DIR = folder


print(device)

In [ ]:
SEED = 42

# saved runs are always used; True also trains whatever is missing, which can take hours on a GPU
ALLOW_TRAINING = False

# cross-fold validation (section 3)
N_FOLDS = 5
MAX_DISTANCE = 4 # max differing bits for two images to be duplicates

CV_PARAMS = {
    "seed": SEED,
    "num_workers": 4 if device.type == "cuda" else 0, # 8 ran out of memory on Windows, each worker loads CUDA
    "pin_memory": device.type == "cuda",

    "batch_size": 8,
    "epochs": 8, # train.py uses 50, 8 for sake of time
    "accumulation_steps": 4,

    "learning_rate": 1e-3,
    "weight_decay": 1e-4,

    "model_name": "not saved",
}

# each setting's folds are saved in models/cross_validation/<name>/fold_<n>
CV_RUNS = {
    "base": {},
    "weight_decay_1e-3": {"weight_decay": 1e-3},
}

# sample of each fold so all 5 fit in about an hour on a GPU, None uses the whole fold
CV_TRAIN_IMAGES = 3000
CV_VAL_IMAGES = 1000

# hyperparameter tuning (section 5)
# settings of scripts/training/train.py and run 20261028_231315
BASE = {"learning_rate": 1e-3, "weight_decay": 1e-4, "backbone_lr_scale": 0.1, "augmentation": True}

# each changes one setting from BASE, {} is BASE as the reference
EXPERIMENTS = [
    {},
    {"learning_rate": 3e-4}, {"learning_rate": 1e-4},
    {"learning_rate": 2e-3}, {"learning_rate": 3e-3}, # 1e-3 was the highest rate tried on  Mac and still won
    {"weight_decay": 1e-5}, {"weight_decay": 1e-3},
    {"backbone_lr_scale": 1.0},
    {"augmentation": False},
]

BATCH_SIZE = 8
ACCUMULATION_STEPS = 4 # 32 images per step; same as train.py
EPOCHS = 8
TRAIN_IMAGES = 3000
VAL_IMAGES = 1000
VAL_FOLD = 0
NUM_WORKERS = 4 if device.type == "cuda" else 0

# final model (section 4)
FINAL_MODEL = "20261029_154330" # 50 epochs, random zoom-ins and flips, best validation mAP50-95

## 1. The Data Set

*Code: Ciaran Whelan. Report: Exploratory Data Analysis.*

**Paste here (Ciarán):**
- `notebooks/data_books/metadata.ipynb`
- `notebooks/data_books/01_data_distribution.ipynb`
- `notebooks/data_books/02_bounding_boxes.ipynb`
- `notebooks/data_books/03_image_properties.ipynb`
- `notebooks/data_books/04_correlation.ipynb`

Keep the same names and remove the `from src` imports.

In [ ]:
df = pd.read_csv(CSV_DIR / "data/metadata.csv")
df["boxes"] = df["boxes"].apply(lambda value: parse_boxes(json.loads(value)))
df.groupby("split").size()

### Pre-processing

*Code: Ciaran Whelan. Report: Preprocessing.*

**Paste here (Ciarán):**
- `src/config/types.py`
- `src/utils/`
- `src/data/preprocessing/`

Keep the same names and remove the `from src` imports.

## 2. The Network Structure

*Code: Emma O'Reilly and Ciaran Whelan. Report: Network Structure.*

**Paste here (Emma, Ciarán):**
- `src/fine_tune/models/resnet.py`
- `src/fine_tune/models/hybrid.py`

Keep the same names and remove the `from src` imports.

## 3. Loss Function, Optimiser and Cross-Fold Validation

### Loss function and optimiser

*Code: Emma O'Reilly and Ciaran Whelan. Report: Loss Function, Optimiser.*

**Paste here (Emma, Ciarán):**
- `src/fine_tune/losses/`
- `src/fine_tune/train.py`

Keep the same names and remove the `from src` imports.

### Cross-fold validation

*Code: Hersh Singh. Report: Cross-Validation and Model Comparison.*

Splits train and val into 5 folds with near-duplicate images kept in the same fold, then trains once per fold. The folds are saved to `results/csvs/data/folds.csv`, and later runs check they come out the same.

In [ ]:
CV_DIR = CSV_DIR / "cross_validation"
CV_PLOT_DIR = PLOT_DIR / "cross_validation"
CV_DIR.mkdir(parents=True, exist_ok=True)
CV_PLOT_DIR.mkdir(parents=True, exist_ok=True)


def image_hash(path):
    img = Image.open(path)
    img.draft("L", (64, 64)) # skips decoding the full resolution
    pixels = np.asarray(img.convert("L").resize((9, 8)), dtype=np.int16)
    return (pixels[:, 1:] > pixels[:, :-1]).flatten() # 64 bits, checks if pixel is brighter than left pixel


def hash_distances(a, b):
    # differing bits between every pair of images in a and b
    a, b = a.astype(np.float32), b.astype(np.float32)
    return a @ (1 - b).T + (1 - a) @ b.T


def near_duplicate_groups(hashes, max_distance):
    # union-find, the top of each chain is the group id
    parent = list(range(len(hashes)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]] # shortcut long chains
            i = parent[i]
        return i

    # 2000 rows at a time, 17k x 17k matrix needs over 1 GB otherwise
    for start in range(0, len(hashes), 2000):
        close = hash_distances(hashes[start:start + 2000], hashes) <= max_distance
        for i, j in zip(*np.nonzero(close)):
            parent[find(start + i)] = find(j) # join two groups so near-duplicates share one group id

    return np.array([find(i) for i in range(len(hashes))])


def assign_folds(groups, n_folds, seed):
    # biggest groups first into folds with least images to balance
    group_ids, sizes = np.unique(groups, return_counts=True)
    order = np.random.default_rng(seed).permutation(len(group_ids)) # random order for equal groups
    order = order[np.argsort(-sizes[order], kind="stable")]

    fold_sizes = np.zeros(n_folds, dtype=int)
    group_fold = {}
    for k in order:
        fold = fold_sizes.argmin() # smallest fold takes next group, keeps folds close in size
        group_fold[group_ids[k]] = fold
        fold_sizes[fold] += sizes[k]

    return np.array([group_fold[group] for group in groups])

In [ ]:
# test is kept for the final model
pool = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
hashes = np.stack([image_hash(DATASET_DIR / path) for path in pool["path"]])

# too strict misses copies, too loose chains everything into one group
rows = []
for distance in [0, 2, 4, 6, 8]:
    sizes = np.unique(near_duplicate_groups(hashes, distance), return_counts=True)[1]
    rows.append({
        "max_distance": distance,
        "groups": len(sizes),
        "images_with_a_duplicate": sizes[sizes > 1].sum(),
        "largest_group": sizes.max(),
        "largest_group_share": sizes.max() / len(pool),
    })

pd.DataFrame(rows).round(3)

In [ ]:
pool["group"] = near_duplicate_groups(hashes, MAX_DISTANCE)
pool["fold"] = assign_folds(pool["group"].to_numpy(), N_FOLDS, SEED)

# every group must sit in exactly one fold
assert (pool.groupby("group")["fold"].nunique() == 1).all() # group split over two folds would leak near-duplicates between train and val

folds_path = CSV_DIR / "data/folds.csv"
if folds_path.exists():
    # the saved folds were used by every training run, so the recomputed ones must match
    saved = pd.read_csv(folds_path).set_index("path")
    assert (saved.loc[pool["path"], "fold"].to_numpy() == pool["fold"].to_numpy()).all()
else:
    pool[["path", "split", "group", "fold"]].to_csv(folds_path, index=False)

# 5 images from each of the 3 biggest groups, one group per row
fig, axes = plt.subplots(3, 5, figsize=(15, 7))
for row, group in zip(axes, pool["group"].value_counts().index[:3]):
    for ax, path in zip(row, pool.loc[pool["group"] == group, "path"].sample(5, random_state=SEED)):
        ax.imshow(Image.open(DATASET_DIR / path))
        ax.set_title(Path(path).name, fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
pool["smoke_boxes"] = pool["boxes"].apply(lambda boxes: sum(box.category == 0 for box in boxes))
pool["fire_boxes"] = pool["boxes"].apply(lambda boxes: sum(box.category == 1 for box in boxes))
pool["source"] = pool["path"].str.extract(r"/([A-Za-z]+)\d+\.jpg$", expand=False)

# folds should have a similar size, class mix and sources
balance = pool.groupby("fold").agg(
    images=("path", "size"),
    groups=("group", "nunique"),
    no_boxes=("label", lambda label: (label == 0).mean()),
    smoke_boxes=("smoke_boxes", "sum"),
    fire_boxes=("fire_boxes", "sum"),
)
balance = balance.join(pd.crosstab(pool["fold"], pool["source"], normalize="index"))
balance.to_csv(CV_DIR / "fold_balance.csv")
balance.round(3)

In [ ]:
# how much the original splits leak
train_groups = set(pool.loc[pool["split"] == "train", "group"])
val = pool[pool["split"] == "val"]
print(f"{val['group'].isin(train_groups).sum()} of {len(val)} original val images share a group with a train image")

test = df[df["split"] == "test"]
test_hashes = np.stack([image_hash(DATASET_DIR / path) for path in test["path"]])
leaked = hash_distances(test_hashes, hashes).min(axis=1) <= MAX_DISTANCE # leaked if train or val image is within MAX_DISTANCE bits
print(f"{leaked.sum()} of {len(test)} test images have a near-duplicate in train or val")

In [ ]:
def load_runs(folder):
    runs = {}
    for run_dir in sorted(folder.glob("*/history.csv")):
        with open(run_dir.parent / "metadata.json") as f:
            runs[run_dir.parent.name] = (json.load(f), pd.read_csv(run_dir))
    return runs


def to_records(split_df, n_images):
    if n_images is not None:
        split_df = split_df.sample(n_images, random_state=SEED)
    return list(split_df[["path", "boxes"]].itertuples(index=False, name=None))


def train_fold(fold, params):
    # same augmentation as scripts/training/train.py
    train_dataset = ModelData(
        DATASET_DIR, to_records(pool[pool["fold"] != fold], CV_TRAIN_IMAGES), train_transform,
        flip_prob=0.5,
        zoom_prob=0.5, zoom_max=1.3
    )
    val_dataset = ModelData(DATASET_DIR, to_records(pool[pool["fold"] == fold], CV_VAL_IMAGES), eval_transform)

    loader_args = dict(
        batch_size=params["batch_size"],
        collate_fn=collate_fn,
        num_workers=params["num_workers"],
        pin_memory=params["pin_memory"],
    )
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_args)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)

    # same random head and shuffle order every fold, but different train/val split
    torch.manual_seed(SEED)
    model = HybridModel().to(device)

    optimiser = AdamW(
        [
            # pretrained backbone gets smaller steps so the ImageNet features arent wiped out
            {"params": model.backbone.parameters(), "lr": params["learning_rate"] * 0.1},
            {"params": [p for name, p in model.named_parameters() if not name.startswith("backbone.")]},
        ],
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"]
    )

    training_objs = TrainingObjects(
        model=model,
        optimizer=optimiser,
        criterion=YOLOLoss(model),
        scaler=torch.amp.GradScaler("cuda", enabled=device.type == "cuda"), # scales loss up so small float16 gradients don't round to zero
        device=device,
        accumulation_steps=params["accumulation_steps"], # 4 batches of 8 add before each step, same as batch of 32
    )

    return Trainer(objs=training_objs).train_model(
        epochs=params["epochs"],
        train_loader=train_loader,
        val_loader=val_loader
    )

In [ ]:
for name, changes in CV_RUNS.items():
    params = CV_PARAMS | changes
    run_dir = SAVED_MODELS / "cross_validation" / name
    done = [meta["notes"][0] for meta, _ in load_runs(run_dir).values() if meta["notes"][-1] == "completed"]
    missing = [fold for fold in range(N_FOLDS) if f"fold {fold} of {N_FOLDS}" not in done]

    if not missing:
        print(f"{name}: all {N_FOLDS} folds saved in models/cross_validation/{name}, using them")
        continue
    if not ALLOW_TRAINING:
        print(f"{name}: folds {missing} are not saved, set ALLOW_TRAINING = True to train them")
        continue

    save_runs_to(run_dir)
    for fold in missing:
        fold_note = f"fold {fold} of {N_FOLDS}"
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results = train_fold(fold, params)

        model_info = {
            "timestamp": timestamp,
            "notes": [
                fold_note,
                f"train images {CV_TRAIN_IMAGES or 'all'}, val images {CV_VAL_IMAGES or 'all'}",
                "completed" if results.error is None else f"failed: {results.error!r}",
            ],
            "git_commit": git_commit,
            "parameters": params,
        }

        results.best_state = None # no model.pt
        Trainer.save_results(results, model_info, f"fold_{fold}")

In [ ]:
cv_tables = {}
for setting in CV_RUNS:
    runs = {name: run for name, run in load_runs(SAVED_MODELS / "cross_validation" / setting).items() if run[0]["notes"][-1] == "completed"}
    if not runs:
        print(f"{setting}: no completed folds yet, set ALLOW_TRAINING = True to train them")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    rows = []
    for name, (meta, history) in runs.items():
        fold = meta["notes"][0]
        line, = axes[0].plot(history["epoch"], history["train_loss"], label=f"{fold} train")
        axes[0].plot(history["epoch"], history["val_loss"], "--", color=line.get_color(), label=f"{fold} val")
        axes[1].plot(history["epoch"], history["map50_95"], marker="o", label=fold)

        best = history.loc[history["map50_95"].idxmax()]
        rows.append({"fold": fold, "run": name, **best[["epoch", "map50", "map50_95", "recall50", "mar_100"]]})

    axes[0].set_title("Loss (dashed val)")
    axes[1].set_title("Validation mAP50-95 on the held-out fold")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(linestyle="--", alpha=0.5)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(CV_PLOT_DIR / f"{setting}_folds.svg")
    plt.show()

    # best epoch per fold, then mean and std
    table = pd.DataFrame(rows).set_index("fold")
    metrics = table[["map50", "map50_95", "recall50", "mar_100"]]
    table = pd.concat([table, metrics.agg(["mean", "std"])])
    table.to_csv(CV_DIR / f"{setting}_folds.csv")
    print(setting)
    display(table.round(4))
    cv_tables[setting] = table

In [ ]:
# paired: both settings are scored on the same five held-out folds
assert len(cv_tables) == 2, "both cross-validation runs need all their folds"
per_fold = pd.concat({setting: table["map50_95"].iloc[:N_FOLDS] for setting, table in cv_tables.items()}, axis=1)
per_fold["difference"] = per_fold["weight_decay_1e-3"] - per_fold["base"]
test_result = stats.ttest_rel(per_fold["weight_decay_1e-3"], per_fold["base"]) # how likely fold-by-fold difference is just noise

display(per_fold.round(4))
print(f"mean difference {per_fold['difference'].mean():+.4f}, paired t-test p = {test_result.pvalue:.2f}")

## 4. Results and Evaluation

### Final model training and test evaluation

*Code: Ciaran Whelan. Report: Results and Evaluation.*

**Paste here (Ciarán):**
- `scripts/training/train.py`
- `scripts/training/evaluate.py`

Keep the same names and remove the `from src` imports.

### Test scores with and without near-duplicates

*Code: Hersh Singh. Report: Near-Duplicates in the Test Split.*

Section 3 found test images with a near-duplicate in train or val. The final model is run once over the test split, keeping each image's detections, so the test set can be scored with and without those images.

In [ ]:
weights = SAVED_MODELS / FINAL_MODEL / "model.pt"
assert weights.exists(), f"{weights} is missing, train the final model in the section above first"

model = HybridModel().to(device)
model.load_state_dict(torch.load(weights, map_location=device, weights_only=True)) # reuse trained weights instead of training final model again
model.eval()

test_df = test.reset_index(drop=True)
test_df["near_duplicate"] = leaked

loader_args = dict(batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
test_loader = DataLoader(ModelData(DATASET_DIR, to_records(test_df, None), eval_transform), shuffle=False, **loader_args)

# Trainer.evaluate only returns the totals, so the detections are kept here per image
predictions, ground_truth = [], []
with torch.no_grad():
    for images, targets in test_loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            outputs = model(images.to(device))
        for det in non_max_suppression(outputs, conf_thres=0.001, iou_thres=0.7, max_det=300): # drop overlapping duplicate boxes, same thresholds Ultralytics validates with
            predictions.append({"boxes": det[:, :4].float().cpu(), "scores": det[:, 4].float().cpu(), "labels": det[:, 5].long().cpu()})
        ground_truth += Trainer._ground_truth(targets, len(images), images.shape[-2:])

In [ ]:
def score(mask):
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True, max_detection_thresholds=[1, 10, 100])
    metric.update([p for p, keep in zip(predictions, mask) if keep], [g for g, keep in zip(ground_truth, mask) if keep]) # only images in this subset are scored
    result = metric.compute()
    return {
        "images": int(np.sum(mask)),
        "mAP50": result["map_50"].item(),
        "mAP50-95": result["map"].item(),
        "smoke mAP50-95": result["map_per_class"][0].item(),
        "fire mAP50-95": result["map_per_class"][1].item(),
        "mAR100": result["mar_100"].item(),
    }


OUT_CSV = CSV_DIR / "evaluation"
OUT_CSV.mkdir(parents=True, exist_ok=True)

near = test_df["near_duplicate"].to_numpy()
subsets = pd.DataFrame({
    "all test images": score(np.ones(len(test_df), dtype=bool)),
    "no near-duplicate in train/val": score(~near),
    "near-duplicate in train/val": score(near),
}).T
subsets.to_csv(OUT_CSV / "test_near_duplicates.csv")
display(subsets.round(4))

## 5. Independent Experiments: Hyperparameter Tuning

*Code: Hersh Singh. Report: Hyperparameter Tuning.*

Each experiment changes one setting from `BASE` and validates on fold 0 of `results/csvs/data/folds.csv` (made in section 3).

In [ ]:
# one folder per experiment, named by what it changes from BASE
TUNING_RUN_DIR = SAVED_MODELS / "hyperparameter_tuning"

TUNING_DIR = CSV_DIR / "hyperparameter_tuning"
TUNING_PLOT_DIR = PLOT_DIR / "hyperparameter_tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)
TUNING_PLOT_DIR.mkdir(parents=True, exist_ok=True)


def load_runs(folder):
    runs = {}
    for run_dir in sorted(folder.iterdir()):
        if not (run_dir / "history.csv").exists():
            continue

        with open(run_dir / "metadata.json") as f:
            meta = json.load(f)

        # older runs prefix the metrics with val_
        history = pd.read_csv(run_dir / "history.csv")
        history = history.rename(columns=lambda c: c if c == "val_loss" else c.removeprefix("val_"))
        runs[run_dir.name] = (meta, history)
    return runs


def short_label(note):
    # name a run by what it changes from BASE, so the reference run reads "base"
    parts = note.split(", val fold")[0].split(", ")
    base = [
        f"learning rate {BASE['learning_rate']:g}", f"weight decay {BASE['weight_decay']:g}",
        f"backbone lr x{BASE['backbone_lr_scale']:g}", f"augmentation {'on' if BASE['augmentation'] else 'off'}",
    ]
    if len(parts) != len(base):
        return note # older runs have free text notes
    return ", ".join(p for p, b in zip(parts, base) if p != b) or "base"


def plot_runs(runs, path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for name, (meta, history) in runs.items():
        batch_size = meta["parameters"]["batch_size"]
        label = short_label(meta["notes"][0])
        line, = axes[0].plot(history["epoch"], history["train_loss"] / batch_size)
        axes[0].plot(history["epoch"], history["val_loss"] / batch_size, "--", color=line.get_color())
        axes[1].plot(history["epoch"], history["map50_95"], marker="o", color=line.get_color(), label=label)

    axes[0].set_title("Loss per image (solid train, dashed val)")
    axes[1].set_title("Validation mAP50-95")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(linestyle="--", alpha=0.5)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(path)
    plt.show()


def summarise(runs):
    rows = []
    for name, (meta, history) in runs.items():
        best = history.loc[history["map50_95"].idxmax()]
        rows.append({
            "run": name,
            "experiment": meta["notes"][0],
            "notes": ", ".join(meta["notes"]),
            "learning_rate": meta["parameters"]["learning_rate"],
            "weight_decay": meta["parameters"]["weight_decay"],
            "epochs": len(history),
            "best_epoch": int(best["epoch"]),
            "best_map50": best["map50"],
            "best_map50_95": best["map50_95"],
            "best_recall50": best.get("recall50"),
            "best_mar_100": best.get("mar_100"),
        })
    return pd.DataFrame(rows)


# the full-length runs in results/models, for comparison
historical = load_runs(SAVED_MODELS)
plot_runs(historical, TUNING_PLOT_DIR / "full_length_runs.svg")

historical_table = summarise(historical)
historical_table.to_csv(TUNING_DIR / "full_length_runs.csv", index=False)
historical_table.round(4)

In [ ]:
tuning_df = df.merge(pd.read_csv(CSV_DIR / "data/folds.csv")[["path", "fold"]], on="path")

train_df = tuning_df[tuning_df["fold"] != VAL_FOLD]
val_df = tuning_df[tuning_df["fold"] == VAL_FOLD]
print(f"{len(train_df)} train and {len(val_df)} val images to sample from")

train_data = list(train_df.sample(TRAIN_IMAGES, random_state=SEED)[["path", "boxes"]].itertuples(index=False, name=None))
val_data = list(val_df.sample(VAL_IMAGES, random_state=SEED)[["path", "boxes"]].itertuples(index=False, name=None))

train_dataset = ModelData(
    DATASET_DIR, train_data, train_transform,
    flip_prob=0.5,
    zoom_prob=0.5, zoom_max=1.3
)
val_dataset = ModelData(DATASET_DIR, val_data, eval_transform)

# seeded in train_experiment so every run sees the same image order
generator = torch.Generator()
loader_args = dict(batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_args)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)

In [ ]:
torch.manual_seed(SEED)
initial_state = HybridModel().state_dict()


def describe(settings):
    return (
        f"learning rate {settings['learning_rate']:g}, weight decay {settings['weight_decay']:g}, "
        f"backbone lr x{settings['backbone_lr_scale']:g}, augmentation {'on' if settings['augmentation'] else 'off'}"
    )


def folder_name(settings):
    # readable run folder, named by what the experiment changes from BASE
    fmt = lambda value: f"{value:.0e}".replace("e-0", "e-")
    changed = []
    if settings["learning_rate"] != BASE["learning_rate"]:
        changed.append(f"lr_{fmt(settings['learning_rate'])}")
    if settings["weight_decay"] != BASE["weight_decay"]:
        changed.append(f"weight_decay_{fmt(settings['weight_decay'])}")
    if settings["backbone_lr_scale"] != BASE["backbone_lr_scale"]:
        changed.append(f"backbone_lr_x{settings['backbone_lr_scale']:g}")
    if settings["augmentation"] != BASE["augmentation"]:
        changed.append(f"augmentation_{'on' if settings['augmentation'] else 'off'}")
    return "_".join(changed) or "base"


def train_experiment(settings):
    model = HybridModel().to(device)
    model.load_state_dict(initial_state)

    optimiser = AdamW(
        [
            # pretrained backbone gets smaller steps so the ImageNet features arent wiped out
            {"params": model.backbone.parameters(), "lr": settings["learning_rate"] * settings["backbone_lr_scale"]},
            {"params": [p for name, p in model.named_parameters() if not name.startswith("backbone.")]},
        ],
        lr=settings["learning_rate"],
        weight_decay=settings["weight_decay"]
    )

    training_objs = TrainingObjects(
        model=model,
        optimizer=optimiser,
        criterion=YOLOLoss(model),
        scaler=torch.amp.GradScaler("cuda", enabled=device.type == "cuda"),
        device=device,
        accumulation_steps=ACCUMULATION_STEPS,
    )

    # workers restart each epoch, so they pick these up
    train_dataset.flip_prob = 0.5 if settings["augmentation"] else 0.0
    train_dataset.zoom_prob = 0.5 if settings["augmentation"] else 0.0

    torch.manual_seed(SEED)
    generator.manual_seed(SEED) # same shuffle order for every experiment, only setting changes

    return Trainer(objs=training_objs).train_model(epochs=EPOCHS, train_loader=train_loader, val_loader=val_loader)

In [ ]:
def experiment_note(settings):
    return f"{describe(settings)}, val fold {VAL_FOLD}, train images {TRAIN_IMAGES}, val images {VAL_IMAGES}"


done = [meta["notes"][0] for meta, _ in load_runs(TUNING_RUN_DIR).values() if "completed" in meta["notes"]] if TUNING_RUN_DIR.exists() else []
missing = [changes for changes in EXPERIMENTS if experiment_note(BASE | changes) not in done]

if not missing:
    print(f"all {len(EXPERIMENTS)} experiments saved in models/hyperparameter_tuning, using them")
elif not ALLOW_TRAINING:
    print(f"{len(missing)} experiments are not saved, set ALLOW_TRAINING = True to train them")
else:
    save_runs_to(TUNING_RUN_DIR)
    for changes in missing:
        settings = BASE | changes
        experiment = experiment_note(settings)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        print(describe(settings))
        results = train_experiment(settings)

        if results.error is None:
            best = results.history.eval[results.best_epoch - 1]
            print_evaluation(best)
            notes = [experiment, "completed", f"[smoke, fire] mAP50-95 {np.round(best.evaluation.map_per_class, 4).tolist()}"]
        else:
            notes = [experiment, f"failed: {results.error!r}"]

        model_info = {
            "timestamp": timestamp,
            "notes": notes,
            "git_commit": git_commit,
            "parameters": {
                "seed": SEED, "num_workers": NUM_WORKERS, "pin_memory": device.type == "cuda",
                "batch_size": BATCH_SIZE, "epochs": EPOCHS, "accumulation_steps": ACCUMULATION_STEPS,
                "learning_rate": settings["learning_rate"], "weight_decay": settings["weight_decay"],
                "model_name": "not saved",
            },
        }

        results.best_state = None # no model.pt
        Trainer.save_results(results, model_info, folder_name(settings))

In [ ]:
if TUNING_RUN_DIR.exists():
    candidates = load_runs(TUNING_RUN_DIR)
    plot_runs(candidates, TUNING_PLOT_DIR / "experiments.svg")

    table = summarise(candidates).sort_values("best_map50_95", ascending=False)
    table.to_csv(TUNING_DIR / "experiments.csv", index=False)
    display(table.round(4))
else:
    print("No runs yet, set ALLOW_TRAINING = True to train them")

In [ ]:
if TUNING_RUN_DIR.exists():
    completed = table[table["notes"].str.contains("completed")]
    best = completed["best_map50_95"].iloc[0] if len(completed) else 0.0
    runner_up = completed["best_map50_95"].iloc[1] if len(completed) > 1 else 0.0

    # one seed and a sample of val images, gaps under 0.01 count as noise
    if best < 0.01:
        print("No winner: every experiment is still near zero mAP, train for longer")
    elif best - runner_up < 0.01:
        print(f"Tie: {completed['experiment'].iloc[0]} is ahead by only {best - runner_up:.4f}")
    else:
        print(f"Best: {completed['experiment'].iloc[0]} (mAP50-95 {best:.4f}, ahead by {best - runner_up:.4f})")